# 第8章 生成模型
## Generative Models — 生成对抗网络（GAN）及其理论

**来源：李宏毅《深度学习教程》第8章 | 对应原书第147-172页**

---

## 一、知识地图：全章结构与脉络

```
第8章 生成模型
├── 8.1 生成对抗网络 (GAN)
│   ├── 8.1.1 生成器 (Generator)
│   │   ├── 为什么需要生成模型？监督学习的"模糊预测"问题
│   │   ├── 从随机分布到复杂分布：输入z的作用
│   │   ├── 无限制生成 vs 条件型生成
│   │   └── 低维向量到高维输出的映射
│   └── 8.1.2 辨别器 (Discriminator)
│       ├── 辨别器的本质：真伪二分类器
│       └── 生成器与辨别器的"内卷"关系
├── 8.2 生成器与辨别器的训练过程
│   ├── 步骤一：固定生成器，训练辨别器
│   ├── 步骤二：固定辨别器，训练生成器
│   └── 交替迭代训练
├── 8.3 GAN的应用案例
│   ├── 动画人物生成
│   ├── 真实人脸生成（渐进式GAN）
│   └── 向量内插与连续变化
├── 8.4 GAN的理论介绍
│   ├── P_G 与 P_data 的分布匹配
│   ├── 判别器目标函数与JS散度的关系
│   └── MinMax博弈的数学本质
├── 8.5 WGAN算法
│   ├── JS散度在高维空间中的问题
│   ├── Wasserstein距离（推土机距离）
│   ├── 1-Lipschitz函数约束
│   ├── Improved WGAN：梯度惩罚
│   └── 谱归一化
├── 8.6 训练GAN的难点与技巧
│   ├── 生成器和辨别器的平衡
│   ├── 用GAN生成文字的困难
│   └── 与VAE、流模型的对比
├── 8.7 GAN的性能评估方法
│   ├── 人眼评估的局限性
│   ├── 图像分类器辅助评估
│   ├── 模式崩塌 (Mode Collapse)
│   ├── 模式丢失 (Mode Dropping)
│   ├── Inception Score
│   └── FID (Frechet Inception Distance)
├── 8.8 条件型生成 (Conditional Generation)
│   ├── 文字到图像的生成
│   ├── 图像翻译 (Pix2Pix)
│   └── 声音到图像的生成
└── 8.9 Cycle GAN
    ├── 无成对数据的风格转换
    ├── 循环一致性损失
    ├── 双向架构
    └── 应用：文字风格转换、无监督翻译、无监督语音识别
```

## 二、为什么需要生成模型？——从监督学习到生成

### 2.1 监督学习的"模糊预测"困境

想象一个场景：我们给小精灵游戏录制视频，让模型预测下一帧画面。使用传统监督学习——输入过去几帧，输出下一帧——训练出来的模型会产生**模糊的、角色消失或出现残影**的画面。

**为什么会这样？** 因为训练数据中存在**一对多映射**：在同一个转角，角色可能向左转，也可能向右转。监督学习面对这两种矛盾答案时的"最优策略"是取其平均——结果就是两个方向的模糊叠加，角色消失。

**核心直觉**：当一个问题有多个正确答案时，监督学习学到的"平均答案"往往是没有意义的。就像问"红眼睛的动漫角色长什么样？"，监督学习会把所有红眼睛角色平均成一张模糊的脸，而生成模型能随机输出其中任何一张清晰的脸。

### 2.2 生成器的本质

生成模型的核心思想是：**给网络注入随机性，让它输出一个分布而非单一值**。

具体做法：
- 从简单分布（如高斯分布 $\mathcal{N}(0, I)$）中采样一个随机向量 $z$
- 将 $z$ 与输入 $x$ 一起送入网络
- 每次采样不同的 $z$，输出就不一样
- 网络整体变成了一个从简单分布到复杂分布的**变换函数**

> **类比**：生成器就像一个画家。给画家一个随机种子（$z$），他能画出不同风格但都合理的画。给定"红眼睛"这个条件，他能画出各种不同的红眼睛角色——有的长发、有的短发、有的戴眼镜。

## 三、GAN的核心思想：生成器 vs 辨别器的博弈

### 3.1 两个网络，一个目标

GAN由两个神经网络组成：

**生成器 $G$**：输入随机向量 $z \sim p_z$（通常是标准正态分布），输出"伪造"的图片 $G(z)$。目标：**骗过辨别器**。

**辨别器 $D$**：输入一张图片（真实的或生成的），输出一个标量 $D(x) \in [0,1]$，表示该图片是"真"的概率。目标：**准确区分真伪**。

### 3.2 训练过程的博弈直觉

训练GAN就像**警察与伪造者的博弈**：
- **第一代**：生成器随机输出噪声 → 辨别器发现"真图片有眼睛，假图片没有" → 轻松分辨
- **第二代**：生成器学会画眼睛 → 骗过第一代辨别器 → 辨别器升级，发现"真图片还有嘴巴" → 再次能分辨
- **第三代**：生成器学会画嘴巴 → 辨别器继续升级 → ...
- 如此反复，生成器越画越逼真，辨别器越来越"挑剔"

### 3.3 训练算法（详细步骤）

**步骤一：固定生成器 $G$，训练辨别器 $D$**
1. 从真实数据分布 $p_{data}$ 中采样 $m$ 张真实图片 $\{x^{(1)}, \ldots, x^{(m)}\}$
2. 从先验分布 $p_z$（如高斯分布）中采样 $m$ 个噪声向量 $\{z^{(1)}, \ldots, z^{(m)}\}$
3. 用生成器生成 $m$ 张假图片 $\{G(z^{(1)}), \ldots, G(z^{(m)})\}$
4. 更新 $D$ 的参数，最大化目标函数：

$$\max_D \frac{1}{m}\sum_{i=1}^{m}\left[\log D(x^{(i)}) + \log(1 - D(G(z^{(i)})))\right]$$

**步骤二：固定辨别器 $D$，训练生成器 $G$**
1. 从先验分布 $p_z$ 中采样 $m$ 个噪声向量 $\{z^{(1)}, \ldots, z^{(m)}\}$
2. 更新 $G$ 的参数，最小化损失（或使用梯度上升最大化）：

$$\min_G \frac{1}{m}\sum_{i=1}^{m}\log(1 - D(G(z^{(i)})))$$

等价于最大化 $\frac{1}{m}\sum_{i=1}^{m}\log D(G(z^{(i)}))$（实践中更常用，因为梯度更好）

交替重复步骤一和步骤二，直到收敛。

In [ ]:
# ============================================
# 第8章 PyTorch示例1：基础GAN的训练框架
# ============================================
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# 超参数
latent_dim = 100       # 输入噪声z的维度
image_size = 28*28     # MNIST图片展平后的维度
batch_size = 128
lr = 0.0002

# 生成器：从100维噪声 -> 784维图片
class Generator(nn.Module):
    def __init__(self, latent_dim, img_dim):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2),
            nn.Linear(1024, img_dim),
            nn.Tanh()  # 输出范围[-1, 1]
        )
    
    def forward(self, z):
        return self.model(z)

# 辨别器：从784维图片 -> 1维真伪分数
class Discriminator(nn.Module):
    def __init__(self, img_dim):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(img_dim, 512),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )
    
    def forward(self, img):
        return self.model(img)

G = Generator(latent_dim, image_size)
D = Discriminator(image_size)
print(f"生成器参数量: {sum(p.numel() for p in G.parameters()):,}")
print(f"辨别器参数量: {sum(p.numel() for p in D.parameters()):,}")
print("\nGAN的两个核心组件已定义完毕。")

In [ ]:
# ============================================
# GAN训练循环框架
# ============================================
criterion = nn.BCELoss()
optimizer_G = optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
optimizer_D = optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))

def train_step_GAN(G, D, real_imgs, optimizer_G, optimizer_D, latent_dim, device='cpu'):
    """GAN的单步训练：先训练D，再训练G"""
    b_size = real_imgs.size(0)
    
    # ---- 步骤一：训练辨别器 ----
    # 真实图片的损失：希望D输出1
    real_pred = D(real_imgs)
    d_loss_real = criterion(real_pred, torch.ones(b_size, 1).to(device))
    
    # 假图片的损失：希望D输出0
    z = torch.randn(b_size, latent_dim).to(device)
    fake_imgs = G(z)
    fake_pred = D(fake_imgs.detach())  # detach阻断梯度流向G
    d_loss_fake = criterion(fake_pred, torch.zeros(b_size, 1).to(device))
    
    d_loss = d_loss_real + d_loss_fake
    optimizer_D.zero_grad()
    d_loss.backward()
    optimizer_D.step()
    
    # ---- 步骤二：训练生成器 ----
    # 生成器希望判别器认为假图是真的
    z = torch.randn(b_size, latent_dim).to(device)
    fake_imgs = G(z)
    fake_pred = D(fake_imgs)
    g_loss = criterion(fake_pred, torch.ones(b_size, 1).to(device))
    
    optimizer_G.zero_grad()
    g_loss.backward()
    optimizer_G.step()
    
    return d_loss.item(), g_loss.item()

print("训练步骤已定义。")
print("关键：步骤一用detach()阻断梯度流向G；步骤二更新G使D相信假图为真。")

## 四、GAN的理论基础——分布匹配的数学本质

### 4.1 训练目标：让 $P_G$ 逼近 $P_{data}$

生成器的输入 $z$ 来自一个已知的简单分布（如高斯分布），通过网络 $G$ 的变换，输出服从一个复杂分布 $P_G$。我们手中有真实数据，它们服从分布 $P_{data}$。

**训练的目标**：找到最优的生成器参数，使得 $P_G$ 和 $P_{data}$ 之间的距离最小：

$$G^* = \arg\min_G \; \text{Div}(P_G, P_{data})$$

其中 $\text{Div}$ 是衡量两个分布差异的某种距离/散度。

### 4.2 判别器如何衡量分布差异

我们无法直接计算 $P_G$ 和 $P_{data}$ 的PDF，但可以分别从中**采样**。GAN的巧妙之处在于：**用判别器来间接衡量两个分布的差异**。

判别器的目标函数（最大化）：

$$V(D, G) = \mathbb{E}_{y \sim P_{data}}[\log D(y)] + \mathbb{E}_{y \sim P_G}[\log(1 - D(y))]$$

**关键结论**：在最优判别器 $D^*$ 下，目标函数的最大值与 $P_G$ 和 $P_{data}$ 之间的**JS散度**直接相关：

$$\max_D V(D, G) = -\log 4 + 2 \cdot JS(P_{data} \| P_G)$$

其中：
- $KL(P\|Q) = \sum_x P(x)\log\frac{P(x)}{Q(x)}$（KL散度，衡量两个分布的非对称差异）
- $JS(P\|Q) = \frac{1}{2}KL(P\|\frac{P+Q}{2}) + \frac{1}{2}KL(Q\|\frac{P+Q}{2})$（JS散度，对称化版本）

### 4.3 GAN的MinMax博弈

综合起来，GAN的训练是一个**极小极大博弈**：

$$\min_G \max_D V(D, G) = \mathbb{E}_{x \sim p_{data}}[\log D(x)] + \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]$$

- **内层**（$\max_D$）：判别器尽力区分真假
- **外层**（$\min_G$）：生成器尽力"愚弄"最优判别器

> **博弈论类比**：这就像围棋中的极小极大搜索——你先假设对手会做出最优的应对，然后在这个前提下选择对自己最有利的策略。

### 4.4 为什么目标函数与交叉熵有关

目标函数 $V(D, G)$ 本质上是**负的交叉熵**。最大化 $V$ 等同于最小化交叉熵，等同于训练一个二分类器（真实数据=类别1，生成数据=类别2）。这让GAN的训练可以用标准的二分交叉熵损失来实现。

## 五、JS散度的失败与WGAN的诞生

### 5.1 JS散度在高维空间中的致命缺陷

**问题一：两个分布几乎不可能重叠**

在高维空间中，图片是低维流形。以二维空间为例，$P_G$ 和 $P_{data}$ 就像是二维空间中的两条直线——除非完全重合，否则它们的交集几乎为零。

另一个角度：我们只是从分布中采样了有限数量的点用于训练。即使两个分布实际上很接近，由于采样点不足，它们也很可能没有任何重叠。

**问题二：不重叠时JS散度恒为常数**

对于任意两个没有重叠部分的分布，$JS(P_G \| P_{data}) = \log 2$，**与分布间的实际距离无关**！

这意味着：
- 生成器稍微变好 → JS散度依然是 $\log 2$ → 判别器看不出差异
- 训练时的损失函数完全无法反映实际进展
- 判别器几乎总能达到100%正确率（只需记住采样过的图片）
- 在WGAN出现之前，训练GAN全靠**人眼看图**来判断好坏

> **类比**：就像用"是否完全重合"来评判两条平行线——无论它们相距1毫米还是1公里，答案永远是"不重合"。这个评判标准完全无效。

### 5.2 Wasserstein距离：推土机距离 (Earth Mover's Distance)

**直觉理解**：假设有两堆土（两个分布），Wasserstein距离就是**用推土机把一堆土移到另一堆土的最小平均移动距离**。

**数学定义**：

$$W(P_r, P_g) = \inf_{\gamma \in \Pi(P_r, P_g)} \mathbb{E}_{(x,y) \sim \gamma}[\|x - y\|]$$

其中 $\Pi(P_r, P_g)$ 是所有边际分布为 $P_r$ 和 $P_g$ 的联合分布的集合，$\inf$ 表示取下确界（在所有"搬运方案"中找最小的那个）。

**为什么Wasserstein距离更好？**

假设 $P_{G0}$ 和 $P_{data}$ 距离为 $d_0$，$P_{G1}$ 和 $P_{data}$ 距离为 $d_1$：

| 指标 | 当距离=$d_0$ | 当距离=$d_1$ | 是否反映差异变化？ |
|------|-------------|-------------|-------------------|
| JS散度 | $\log 2$ | $\log 2$ | **否** |
| Wasserstein | $d_0$ | $d_1$ | **是** |

> **演化类比**：人类眼睛的演化是一个渐进过程——从感光细胞到凹陷的感光区再到充满液体的复杂结构。Wasserstein距离允许这种**逐步改进**，而JS散度则要求一步到位——这是WGAN让训练变得更加平滑和稳定的根本原因。

### 5.3 WGAN的数学形式

通过Kantorovich-Rubinstein对偶定理，Wasserstein距离可以写为：

$$W(P_{data}, P_G) = \max_{\|D\|_L \leq 1} \left(\mathbb{E}_{x \sim P_{data}}[D(x)] - \mathbb{E}_{x \sim P_G}[D(x)]\right)$$

其中约束 $\|D\|_L \leq 1$ 意味着 $D$ 必须是**1-Lipschitz函数**：

$$|D(x_1) - D(x_2)| \leq \|x_1 - x_2\|$$

**直观理解**：$D$ 的输出变化不能超过输入变化的幅度——函数必须"足够平滑"，斜率有上限。

**为什么需要这个约束？** 如果没有Lipschitz约束，判别器可以让真实数据的输出趋向无限大、生成数据的输出趋向无限小，训练永远无法收敛。加上约束后：当两个分布距离近时，判别器无法同时给真实数据极高分数和生成数据极低分数——因为那样变化太剧烈，违反平滑约束。

### 5.4 实现1-Lipschitz约束的三种方法

**方法一：权重裁剪（原始WGAN）**
- 每次更新后将判别器参数限制在 $[-c, c]$ 范围内
- 简单直观，但只是一种粗糙的近似
- 裁剪值 $c$ 的选择很敏感：太大会训练不稳，太小会把判别器逼到线性函数

**方法二：梯度惩罚（WGAN-GP / Improved WGAN）**
- 在判别器损失函数中加入惩罚项：
$$\lambda \cdot \mathbb{E}_{\hat{x}}[(\|\nabla_{\hat{x}} D(\hat{x})\|_2 - 1)^2]$$
- $\hat{x}$ 是在真实样本和生成样本之间随机插值的点：$\hat{x} = \epsilon x_{real} + (1-\epsilon) x_{fake}$，$\epsilon \sim U(0,1)$
- 直觉：要求判别器在真假数据之间的所有路径上的梯度范数都接近1
- 这是目前最常用的方法

**方法三：谱归一化（Spectral Normalization）**
- 将每一层的权重矩阵除以它的谱范数（最大奇异值）
- 保证了严格的1-Lipschitz约束
- 不需要计算二阶梯度，比梯度惩罚更高效

In [ ]:
# ============================================
# PyTorch示例2：WGAN-GP的梯度惩罚实现
# ============================================
import torch
import torch.nn as nn

def compute_gradient_penalty(D, real_samples, fake_samples, device='cpu'):
    """
    计算WGAN-GP的梯度惩罚项
    
    原理详解：
    1. 在真实样本和假样本之间随机插值生成 hat{x}
    2. 计算判别器对 hat{x} 的梯度
    3. 惩罚梯度L2范数偏离1的程度
    
    为什么在插值点计算？最优判别器在P_data和P_G之间
    的直线上，梯度范数几乎处处为1。
    """
    batch_size = real_samples.size(0)
    # 随机插值系数 epsilon ~ U(0, 1)
    epsilon = torch.rand(batch_size, 1, device=device)
    epsilon = epsilon.expand_as(real_samples)
    
    # 插值：hat{x} = epsilon * x_real + (1-epsilon) * x_fake
    interpolates = epsilon * real_samples + (1 - epsilon) * fake_samples
    interpolates.requires_grad_(True)
    
    # 判别器对插值点的输出
    d_interpolates = D(interpolates)
    
    # 计算梯度（需要二阶梯度，create_graph=True）
    gradients = torch.autograd.grad(
        outputs=d_interpolates,
        inputs=interpolates,
        grad_outputs=torch.ones_like(d_interpolates),
        create_graph=True,
        retain_graph=True,
        only_inputs=True
    )[0]
    
    # 梯度惩罚 = 均值((||grad||_2 - 1)^2)
    gradients = gradients.view(batch_size, -1)
    gradient_norm = gradients.norm(2, dim=1)
    gradient_penalty = ((gradient_norm - 1) ** 2).mean()
    
    return gradient_penalty

print("WGAN-GP梯度惩罚已实现。")
print("核心要点:")
print("1. 判别器(Critic)输出不经过Sigmoid——直接输出标量")
print("2. Critic损失 = E[D(fake)] - E[D(real)] + lambda * GP")
print("3. Generator损失 = -E[D(G(z))]")
print("4. lambda通常取10")

## 六、模式崩塌与模式丢失

### 6.1 模式崩塌 (Mode Collapse)

**现象**：生成器反复输出相同的几张图片，缺乏多样性。比如让它生成100张脸，实际只有3-5种不同的脸在重复。

**直观解释**：判别器的"盲点"——生成器发现某张图能稳定骗过判别器，就一直生成它，完全忽略其他可能的模式。

**深入分析**：
- 从博弈论角度看，生成器找到了一个局部均衡——一种最容易骗过判别器的模式
- 判别器没看到足够的"反例"来迫使生成器探索更多模式
- 训练动力学导致生成器困在一个小区域

**如何检测**：让生成器从不同 $z$ 产生一批图片，观察是否有大量重复或极相似的输出。

**缓解方法（没有完美的解决方案）**：
- 保存训练过程中的多个检查点，在模式崩塌前停止
- 小批次判别（Mini-batch Discrimination）：让判别器能察觉整个批次缺乏多样性
- 使用Wasserstein GAN（对模式崩塌有一定缓解）
- 使用展开的GAN（Unrolled GAN）

### 6.2 模式丢失 (Mode Dropping)

**现象**：生成器能产生漂亮且多样的图片，但只能覆盖真实数据分布的**一部分**。

> **具体类比**：真实数据中有猫、狗、鸟三种动物。模式丢失意味着生成器只生成猫和狗——单独看每张图都很好，但它完全"遗忘"了鸟这个类别。你平时看几张图可能觉得"都很好"，只有大量采样对比真实数据分布才能发现这个问题。

**为什么更危险**：比模式崩塌更难检测——生成的图片**看起来**多样，但实际上缺少了对某一整类样本的覆盖能力。即使是SOTA的GAN（BigGAN、Progressive GAN）也或多或少存在这个问题。

**模式丢失在今天**：如果你看多了GAN生成的"真人"脸，会发现虽然非常逼真，但似乎"来来去去就那么几张脸"。这就是模式丢失的证据——真实人类面貌的多样性远大于GAN的覆盖范围。

## 七、GAN性能评估

### 7.1 为什么评估GAN如此困难？

GAN的输出没有"标准答案"——没有哪个像素值一定是"对"的。这与分类任务完全不同。早年的GAN论文甚至**没有数值结果**，只在论文末尾放几张生成的图片就算"证明了效果"。这显然不客观、不可靠、不可比较。

### 7.2 基于图像分类器的评估思路

将生成的图片送入一个预训练的图像分类器（如Inception网络）：
- 如果分类器给出**集中**的概率分布（例如90%确定是猫）→ 图片**质量高**
- 如果分类器给出**均匀**的分布 → 图片质量差（四不像）

但光有质量衡量不够，还需要**多样性**衡量。

### 7.3 Inception Score (IS)

$$\text{IS} = \exp\left(\mathbb{E}_{x \sim P_G} [KL(p(y|x) \| p(y))]\right)$$

- $p(y|x)$：给定图片 $x$，分类器输出的类别分布（越集中=质量越高）
- $p(y)$：所有生成图片的类别边缘分布（越均匀=多样性越高）
- KL散度在两者都好的时候最大
- IS越高越好

### 7.4 Frechet Inception Distance (FID) — 当前最常用

**计算步骤**：
1. 将真实图片和生成图片都送入Inception网络
2. 取出进入Softmax之前最后一层的特征向量（通常上千维）
3. 假设两组特征都服从**多元高斯分布**（这个假设可能不完美，但实践中有效）
4. 计算两个高斯分布之间的Frechet距离：

$$\text{FID} = \|\mu_r - \mu_g\|^2 + \text{Tr}(\Sigma_r + \Sigma_g - 2(\Sigma_r \Sigma_g)^{1/2})$$

- $\mu_r, \Sigma_r$：真实图片特征的均值和协方差矩阵
- $\mu_g, \Sigma_g$：生成图片特征的均值和协方差矩阵
- $\text{Tr}$：矩阵的迹（对角线元素之和）
- FID越小越好（0表示两个分布完全一致）

### 7.5 评估的注意事项与陷阱

- **过拟合欺骗FID**：生成器"背"下训练集 → FID=0 → 看起来完美，但没有泛化能力
- **简单变换**：生成器把所有训练图片左右反转 → FID也很低 → 但实际上什么都没学到
- **需要足够样本**：FID需要大量采样（通常数万张）才能准确估计分布
- 《Are GANs Created Equal?》的发现：在相同架构下，不同GAN变体通过充分调参后表现差异不大
- **评估本身就是一个研究领域**：如何客观、自动地评估生成模型的好坏至今没有完美解决

## 八、条件型生成 (Conditional GAN)

### 8.1 从无条件到有条件

- 无条件GAN：随机噪声 $z \rightarrow G \rightarrow y$（随机生成任意图片）
- 条件型GAN：$(z, x) \rightarrow G \rightarrow y$（根据条件 $x$ 控制生成内容）

**应用示例**：文字→图像。输入"红眼睛、黑头发"，生成对应动漫角色。每次采样不同 $z$ 会产生不同但都符合描述的角色。

### 8.2 关键设计：判别器必须同时看到条件和图片

如果判别器只看 $y$（生成的图片），生成器会**无视条件 $x$**——只要画出好图片就能骗过判别器，为什么要费事关心输入文字？

**解决方案**：判别器输入 $(x, y)$，输出分数表示"图片好**且**与条件匹配"。

**训练数据构造（必须成对）**：
- 正样本：真实图片 + 正确文字描述 → 标签1
- 负样本一：真实图片 + 错误文字描述 → 标签0
- 负样本二：生成图片 + 给定文字描述 → 标签0
- 负样本三：生成图片 + 错误文字描述 → 标签0

实际应用中，往往还需要"好图片但文字不匹配"的负样本来加强训练。

### 8.3 图像翻译 (Pix2Pix) 与更多应用

- Pix2Pix：输入一张图片（如房屋设计图），输出另一张图片（如实景图）
- 条件=输入图片，仍需要成对数据训练
- 纯GAN可能产生输入没有的内容（"创造力过度丰富"），所以通常**GAN + 监督学习（L1损失）** 联合训练
- 其他应用：黑白着色、去雾、白天→夜晚、素描→实景
- 声音→图像：听狗叫声画出一只狗
- 图像→视频：让蒙娜丽莎动起来
- 条件型GAN还可以用GAN+监督学习联合训练达到最好效果

In [ ]:
# ============================================
# PyTorch示例3：条件型GAN的判别器设计
# ============================================
class ConditionalDiscriminator(nn.Module):
    """
    条件型GAN的判别器
    输入：图片 + 条件（如文字描述编码）
    输出：真伪评分
    
    关键：判别器必须同时评估'图片质量'和'条件匹配度'
    如果只看图片不看条件，生成器会无视条件输入。
    """
    def __init__(self, img_dim, condition_dim, hidden_dim=256):
        super().__init__()
        # 图片特征提取
        self.img_layer = nn.Sequential(
            nn.Linear(img_dim, hidden_dim),
            nn.LeakyReLU(0.2)
        )
        # 条件特征提取
        self.cond_layer = nn.Sequential(
            nn.Linear(condition_dim, hidden_dim),
            nn.LeakyReLU(0.2)
        )
        # 联合特征判断
        self.joint_layer = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )
    
    def forward(self, img, condition):
        img_feat = self.img_layer(img)
        cond_feat = self.cond_layer(condition)
        # 融合图片和条件特征
        combined = torch.cat([img_feat, cond_feat], dim=1)
        return self.joint_layer(combined)

print("条件型GAN判别器已定义。")
print("核心设计：img和condition分别编码后融合，联合判断真伪+匹配度。")

## 九、Cycle GAN：无需成对数据的风格转换

### 9.1 问题设定

我们想把真人照片转为动漫头像，但**没有成对数据**——只有一堆真人照片和一堆动漫头像，没有"哪个真人对应哪个动漫"。

直接用GAN的问题：生成器可能无视输入，随便生成一张"好看"的动漫图来糊弄判别器。

### 9.2 Cycle GAN的核心思想

**训练两个生成器**（X域=真人照片，Y域=动漫头像）：
- $G_{X \to Y}$：将X域转为Y域
- $G_{Y \to X}$：将Y域转回X域

**循环一致性损失 (Cycle Consistency Loss)**：

$$\mathcal{L}_{cyc}(G, F) = \mathbb{E}_{x \sim p_X}[\|F(G(x)) - x\|_1] + \mathbb{E}_{y \sim p_Y}[\|G(F(y)) - y\|_1]$$

其中 $G = G_{X\to Y}$，$F = G_{Y\to X}$。

**直觉**：一张真人照片 → 转成动漫 → 再转回来，应该和原来一模一样。

这个约束**强迫** $G_{X \to Y}$ 的输出与输入有强关联——否则 $G_{Y \to X}$ 无法将其还原。

### 9.3 完整损失函数

$$\mathcal{L} = \mathcal{L}_{GAN}(G, D_Y, X, Y) + \mathcal{L}_{GAN}(F, D_X, Y, X) + \lambda \cdot \mathcal{L}_{cyc}(G, F)$$

包含两个GAN损失（每个方向各有一个判别器）加循环一致性损失。$\lambda$ 控制循环损失的权重。

### 9.4 潜在的失败模式与为什么通常不会发生

**理论漏洞**：$G_{X\to Y}$ 可能学会"把图片左右翻转"，$G_{Y\to X}$ 学会"再翻转回来"——循环一致性满足，但实际没有任何有意义的风格转换。

**为什么实践中很少发生**：神经网络非常"懒惰"——给定一张图片，它倾向于输出和输入类似的东西，不太愿意做复杂的转换。所以实际中即使只用普通GAN做风格转换，效果也往往不差。Cycle GAN在大多数情况下能自然地学到有意义的转换。

### 9.5 拓展应用

- **StarGAN**：在多种风格之间转换（不只两种）
- **Disco GAN, Dual GAN**：类似思路
- **文字风格转换**：负面句子→正面句子（输入和输出都是序列，可用Transformer架构）
- **无监督翻译**：用中文和英文句子（无配对数据），机器可能学会翻译
- **无监督语音识别**：声音→文字（无配对数据）

In [ ]:
# ============================================
# PyTorch示例4：Cycle GAN循环一致性损失
# ============================================
import torch.nn.functional as F

def cycle_consistency_loss(G_XY, G_YX, real_X, real_Y):
    """
    Cycle GAN的循环一致性损失
    
    Args:
        G_XY: X->Y的生成器
        G_YX: Y->X的生成器
        real_X: 来自X域的真实样本
        real_Y: 来自Y域的真实样本
    
    Returns:
        总循环一致性损失（前向 + 后向）
    """
    # 前向循环: X -> Y -> X，应该回到原点
    fake_Y = G_XY(real_X)
    reconstructed_X = G_YX(fake_Y)
    forward_loss = F.l1_loss(reconstructed_X, real_X)
    
    # 后向循环: Y -> X -> Y，也应该回到原点
    fake_X = G_YX(real_Y)
    reconstructed_Y = G_XY(fake_X)
    backward_loss = F.l1_loss(reconstructed_Y, real_Y)
    
    return forward_loss + backward_loss

print("Cycle GAN循环一致性损失已实现。")
print()
print("核心思想：x -> G(x) -> F(G(x)) 应该约等于 x")
print("这强制生成器保留输入的结构信息。")
print()
print("为什么使用L1损失而非L2？")
print("L1鼓励更锐利的输出（对模糊的惩罚更小）。")

## 十、训练GAN的难点与技巧

### 10.1 生成器与辨别器的微妙平衡

GAN以难训练闻名。两个网络"互相砥砺才能互相成长"：
- 如果辨别器太强 → 生成器永远骗不过去 → 生成器收不到有用梯度 → 停止进步
- 如果生成器太强 → 辨别器无法区分 → 辨别器停止进步 → 生成器失去目标 → 也停止进步
- 一方停止训练，另一方跟着变差——训练崩溃

### 10.2 用GAN生成文字的特殊困难

**图像GAN**：生成器输出连续值（像素）→ 微小参数变化 → 微小输出变化 → 判别器输出微小变化 → 可以计算梯度。

**文字GAN（离散输出问题）**：
- 生成器输出离散词元（token）的概率分布
- 微小参数变化 → 概率分布微小变化 → 但取argmax时，最大值位置**不变**
- 判别器输入不变 → 输出不变 → 梯度为零 → **无法用梯度下降训练**

**解决方案**：使用强化学习（策略梯度）训练生成器。但RL和GAN都很不稳定，组合在一起挑战巨大。ScratchGAN证明了通过大量调参（批大小>1000，特殊训练技巧等）可以从零训练文本GAN。

### 10.3 与其他生成模型的对比

| 模型 | 优点 | 缺点 | 适用场景 |
|------|------|------|----------|
| GAN | 生成质量最高（清晰度） | 训练难，模式崩塌 | 图像生成 |
| VAE | 训练稳定，有明确损失 | 生成图较模糊 | 连续数据 |
| 流模型 | 精确似然计算 | 计算量大 | 密度估计 |
| 扩散模型 | 质量极高 | 推理慢 | 高质量图像生成 |

### 10.4 能不能用监督学习来生成？

可以为每张图片随机配一个高斯向量作为输入，用监督学习（MSE损失）训练网络。这种方法确实可行（如GLO——Generative Latent Optimization），但如果纯用随机向量训练，结果往往很差，需要特殊技巧。

## 十一、常见误区与注意事项

### 误区1："GAN的损失函数能反映训练进度"

**错误**：GAN没有单调递减的损失函数。判别器损失下降可能只因为判别器变强了，不代表生成器变好。在WGAN出现前，训练GAN就是"盲盒"——每训练几步就得把图片可视化打印出来看，发现结果不好就重新调参。**必须可视化输出图片来监控训练**。

### 误区2："判别器准确率越高越好"

**错误**：如果判别器准确率达到100%，说明两个分布完全不重叠，生成器收不到有用梯度。理想情况是判别器准确率维持在~50%（纳什均衡点）。

### 误区3："Generator和Discriminator应该同步训练"

**部分错误**：实践中判别器通常需要更多训练步数（如WGAN中n_critic=5）。如果判别器跟不上生成器，就无法提供有意义的反馈梯度。

### 误区4："GAN已经过时了，扩散模型取代了一切"

**过于简化**：对于图像生成，GAN仍然是生成速度最快（单步前向）且质量很高的模型。扩散模型需要多次迭代去噪，推理慢得多。在实际部署中各有定位。

### 误区5："Cycle GAN保证了完美的输入输出对应关系"

**实际不是**：理论上有漏洞（如左右翻转也可满足循环一致性），但实践中神经网络"懒惰"，倾向于保持输入结构，所以效果通常不错。但这不是数学上严格保证的。

## 十二、跨章节连接

| 章节 | 连接关系 |
|------|----------|
| **Ch9 扩散模型** | 同为生成模型。扩散模型通过逐步去噪生成，GAN通过博弈一次生成。两者可以结合 |
| **Ch11 自编码器** | VAE是GAN之外的生成模型。自编码器的解码器可直接作为生成器。去噪自编码器与扩散模型的去噪有关 |
| **Ch12 对抗攻击** | 两者都有"对抗"概念。GAN：生成器vs判别器的内部对抗。对抗攻击：攻击者vs模型的外部对抗。思想同源 |
| **Ch13 迁移学习** | 领域对抗训练(DANN)直接借用GAN框架：特征提取器=生成器，领域分类器=判别器 |
| **Ch14 强化学习** | 训练GAN的生成器与训练RL策略函数类似——都需要最大化来自"环境"（判别器/奖励）的分数 |

## 十三、核心要点总结

1. **动机**：当问题有多个正确答案时，监督学习输出"平均答案"导致模糊。生成模型输出分布，可随机采样各种合理答案。

2. **GAN结构**：生成器 $G(z)$ 从噪声生成数据，辨别器 $D(x)$ 区分真伪。两者交替训练，互相促进形成"内卷"式提升。

3. **训练本质**：极小极大博弈 $\min_G \max_D V(D,G)$。$V$ 是负交叉熵，最大化它等同于训练一个二分类器。

4. **JS散度的失败**：高维空间中 $P_G$ 和 $P_{data}$ 几乎不重叠，JS散度恒为 $\log 2$，无法反映分布的实际距离和训练进展。

5. **Wasserstein距离**："推土机距离"能平滑反映分布差异，即使它们不重叠。WGAN用Wasserstein距离替代JS散度，训练稳定得多。

6. **1-Lipschitz约束**：WGAN要求判别器满足 $|D(x_1)-D(x_2)| \leq \|x_1-x_2\|$。可通过梯度惩罚(WGAN-GP)或谱归一化实现。

7. **模式崩塌/丢失**：生成器只输出少数模式（崩塌）或遗漏某些模式（丢失），是GAN的核心未解决问题。

8. **条件型GAN**：判别器同时输入图片和条件，实现可控生成。训练需要成对的标注数据。

9. **Cycle GAN**：通过循环一致性损失，在**无成对数据**下实现风格转换——这是无监督学习的重要范式。

10. **评估方法**：FID是最常用指标，通过比较真实和生成图片在特征空间中的分布距离来衡量。FID越小越好。

11. **训练难点**：GAN极难训练——二网络平衡度微妙，离散输出（文字）尤其困难。调参、看可视化、不断重试曾是训练GAN的常态。

## 十四、练习与思考

1. 为什么监督学习在小精灵游戏视频预测中会产生模糊画面？从数学期望的角度解释。

2. 推导GAN的最优判别器 $D^*(x) = \frac{p_{data}(x)}{p_{data}(x) + p_G(x)}$，并证明此时 $\max_D V(D,G)$ 与JS散度的关系。

3. 解释为什么JS散度在高维空间中几乎总是返回 $\log 2$。"两直线几乎不可能相交"这个直觉的数学根据是什么？

4. 用"推土机"直觉解释Wasserstein距离为何能解决JS散度的问题。为什么"移动土"的思维能给出有意义的梯度？

5. 什么是1-Lipschitz约束？WGAN为什么需要它？如果不需要这个约束，训练会出什么问题？

6. 实现完整的WGAN-GP，在MNIST上训练，比较有/无梯度惩罚时的训练稳定性。

7. 修改条件型GAN判别器，使其能处理"文字描述+图片"的配对评估。设计正负样本构造策略。

8. 解释Cycle GAN的循环一致性损失为何能约束输入-输出关系。如果去掉循环一致性，会发生什么？

9. 模式崩塌和模式丢失有什么不同？各有什么检测方法？为什么模式丢失更危险？

10. FID的计算需要什么假设（高斯分布）？这些假设在什么情况下会被违反？

11. 为什么用GAN生成文字极其困难？从梯度传播（argmax导致梯度为零）的角度分析。